# Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Getting some details about the dataset

**First 5 rows**

In the dataset there are some inconsistencies, like case problems, different date and currency formats, null values. we will explore more in further steps

In [ ]:
# First 5 rows
df=pd.read_csv('Saas_Dataset.csv')
df.head()

In [ ]:
# Count of rows and columns
df.shape

In [ ]:
# Column names
df.columns

In [ ]:
# Details about row count, column count, type and count of null values
df.info()

In [ ]:
# Descriptive statistics, shape of a dataset's distribution
df.describe()

**Value counts for each country**



In [ ]:
df['country'].value_counts()

Above we see 11 countries shown. But in reality, some of them represent the same country, they are just written differently, such as: U.S.A, America, and USA are the same country. In further steps, these will be cleaned. Otherwise, data would be categorized wrong.

**Value counts for each Billing Cycle**

In [ ]:
df['billing_cycle'].value_counts()

In Billing Cycle column, we have 4 categories shown, but in reality we have 2 categories. They are written in different cases: Uppercase, Lowercase, Sentence case

**Value conts for Acquisition Channel**

In [ ]:
df['acquisition_channel'].value_counts()

Same issue here

Value Counts for Plan Level

In [ ]:
df['plan_level'].value_counts()

# Data Cleaning

In [ ]:
# Mapping country names acurately
df['country'] = df['country'].replace({'America': 'United States', 'U.S.A.': 'United States', 'USA':'United States', 'DE':'Germany', 'UK':'United Kingdom', 'U.K.':'United Kingdom'})

# Mapping billing cycle names
df['billing_cycle'] = df['billing_cycle'].replace({'monthly': 'Monthly', 'ANNUAL': 'Annual'})

# Mapping accquisition chanle
df['acquisition_channel'] = df['acquisition_channel'].replace({'google ads': 'Google Ads'})

df.head(3)

Here, i mapped the messy columns to to correct form in the dataset that i mentioned earlier.

**Replacing Unnecessary Columns**

In [ ]:
# Replacing "+" signs
df['last_login_days_ago'] = (df['last_login_days_ago'].str.replace('+', '', regex=False).astype(int))

df['active_days'] = (df['active_days'].str.replace('+', '', regex=False).astype(int))

There was additional signs beside the numbers in last_login_days_age and active_days columns. Removing them allows me to analyze data accurately, and be able to use them in further tasks like calculations, comparisons. Otherwise data type would not be numerical but be object, and that prevents furthe steps.

**Converting signup_date to proper format**

In signup_date column, dates were mixed. I converted them to one format to interpret every date consistently as the correct date instead of misreading or failing to parse some values. This ensures accurate sorting, filtering, and time-based analysis.

In [ ]:
df['signup_date'] = pd.to_datetime(df['signup_date'], format='mixed')
df['signup_date'] = df['signup_date'].dt.strftime('%Y-%m-%d')
df.head(3)

# Feature Engineering

**Replacing "%" sign and converting to percentage**

In feature_usage_score column, there are "%" signs which prevent calculations. Here I removed them, converted type to float and divided by 100 to represent percent values accurately. I changed column name for clarity as well.

In [ ]:
df['feature_usage_score'] = df['feature_usage_score'].str.replace('%', '').astype(float)/100
df.rename(columns={'feature_usage_score': 'feat_usage_score_percent'}, inplace=True)
df.head()

**Converting Currencies**

The monthly_revenue column is the most inconsistent due to the presence of multiple currencies within a single column. If we don't clean it, values from different currencies will be treated as if they have the same unit, leading to incorrect calculations and misleading insights. As a result, totals, averages, comparisons, and visualizations based on revenue will be inaccurate.

First I found how many values contains different currency signs.

In [ ]:
# finding the values in size column which has '£' in it
df['monthly_revenue'].loc[df['monthly_revenue'].str.contains('£', regex=False)].value_counts().sum()

In [ ]:
# finding the values in size column which has '$' in it
df['monthly_revenue'].loc[df['monthly_revenue'].str.contains('$', regex=False)].value_counts().sum()

In [ ]:
# finding the values in size column which has '€' in it
df['monthly_revenue'].loc[df['monthly_revenue'].str.contains('€',regex=False)].value_counts().sum()

After that, i created function to convert all currencies to USD because USD is most used currency worldwide (euro to dollar * 1.08, pound to dollar * 1.27). I removed the spaces, commas,signs  and converted datatypes to float as well

In [ ]:
def convert_currency(currency):
    if isinstance(currency, str):

        # removing spaces and commas
        currency = currency.strip()
        currency = currency.replace(',', '')

        if '$' in currency:
            return float(currency.replace('$', ''))
        elif '€' in currency:
            return float(currency.replace('€', '')) * 1.08
        elif '£' in currency:
            return float(currency.replace('£', '')) * 1.27

    return currency

df['monthly_revenue'] = df['monthly_revenue'].apply(convert_currency)
df.rename(columns={'monthly_revenue': 'monthly_revenue_usd'}, inplace=True)

**Missing values**

In [ ]:
df.isnull().sum()

There is only 1 missing value in company column, so it's okay to drop it

In [ ]:
df.dropna(subset=['company'], inplace=True)

nps_score column has 10 missing values. I converted the nps_score column to a numeric data type. Any values that couldn't be converted (such as text) were replaced with NaN (missing values). Then, I filled the missing values with the median of scores. I used the median because it is less affected by extreme values (outliers) than the mean, and that makes it a more reliable choice for replacing missing numerical data.

In [ ]:
# Convertng 'nps_score' to numeric, coercing errors to NaN
df['nps_score'] = pd.to_numeric(df['nps_score'], errors='coerce')
median_nps_score = df['nps_score'].median()

# Filling NaN values in 'nps_score' with median
df['nps_score'].fillna(median_nps_score, inplace=True)

print(f"Median of nps_score: {median_nps_score}")

In [ ]:
df.isnull().sum()

**Duplicates**

I checked for duplicate values and dropped them. (there were 19)

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()



**Visualization of target distribution**

I visualized churn distribution
churned (1) - customers who left the service
non-chuned (0) - cutomers who stay

In [ ]:
# Target
sns.countplot(x='churned', data=df)
plt.title("Churn Distribution")
plt.show()

**Feature Correlation Heatmap**


In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

Here we can see colmumn dependencies.
For example, nps_score and feature_usage_score columns show high positive correlation (0.66). This means that when the satisfaction level is high, the customer is more likely to use the feature.
churned, last_login_days_ago (0.67) - means more the last login days ago, which means the customer doesn't use the feature frequently, more likely that they will live the service.

feat_usage_score_percent, support_ticketss (-0.58) high negative correlation. This means, the less number of times a customer contacted customer support for assistance, more likely they'll use the service/feature. Which eventually means more likely the customer will not churn

# Outliers

**Log Transformation**

I applied a log transformation (log1p) to the monthly_revenue_usd and support_tickets columns to reduce skewness, minimize the influence of extreme values, and create a more balanced distribution. I used log1p() because it safely handles zero values.

In [ ]:
df["log_monthly_revenue"] = np.log1p(df["monthly_revenue_usd"])
df["log_support_tickets"] = np.log1p(df["support_tickets"])

df["active_days_clipped"] = df["active_days"].clip(0, 30)

I created boxplots to compare the distributions of monthly_revenue_usd and support_tickets before and after applying the log transformation. The plots show that, after the transformation, the data became more compressed and balance. This shows that the log transformation reduced skewness and made the variables more suitable for further analysis.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))

# Revenue
sns.boxplot(y=df["monthly_revenue_usd"], ax=axes[0, 0])
axes[0, 0].set_title("Revenue (Before Log)")

sns.boxplot(y=df["log_monthly_revenue"], ax=axes[1, 0])
axes[1, 0].set_title("Revenue (After Log)")

# Support Tickets
sns.boxplot(y=df["support_tickets"], ax=axes[0, 1])
axes[0, 1].set_title("Support Tickets (Before Log)")

sns.boxplot(y=df["log_support_tickets"], ax=axes[1, 1])
axes[1, 1].set_title("Support Tickets (After Log)")

plt.tight_layout()
plt.show()

# Insights

**Which country has the most customers?**

In [ ]:
df['country'].value_counts()

United States has the most customers (426) which is completely normal, because most of the Saas services are based on US. After US, it's European countris (UK, Germany). And India has the least customers

**Which industry has the most customers?**

In [ ]:
df['industry'].value_counts()

Education and SaaS. This means these industries are in high demand

**customers on each plan level**

In [ ]:
df['plan_level'].value_counts()

most people use free plan level and after that, basic. The reason can be: they are affordable, and ideal for people who do not plan to stay long in the service. Also, most people choose and stay in free option for a while in order to check whether the service is worth to stay.
and Enterprise has least customers, because it's the most expensive, and most people can't afford it.

**Which acquisition channel brings the most customers?**

In [ ]:
df['acquisition_channel'].value_counts()

Google Ads is the acquisition channel that brings in the most customers. One possible reason can be that Google Ads has a broad online reach, allowing businesses to advertise to users while they browse websites or search online. This increased visibility may encourage more potential customers to visit the service.
Direct has thw least customers. A possible explanation is that fewer customers were already familiar with the brand, so most discovered the service through advertising or other acquisition channels instead. (These are only hypotheses, not exact evidence)

**Which billing_cycle has most customers?**

In [ ]:
df['billing_cycle'].value_counts()

Monthly. The reason can be: it's more affordable, and people have an option to leave the service in a few months, if needed

**Count of churned (1) and stayed (0) customers**

In [ ]:
df['churned'].value_counts()

In [ ]:
# Average monthly revenue per customer
df['monthly_revenue_usd'].mean()

**What's the average monthly revenue per plan level?**

In [ ]:
df.groupby('plan_level')['monthly_revenue_usd'].mean().sort_values(ascending=False)

the highest avg revenue comes from enterprise, because it's the most expensive plan level amongst the others

**Which acquisition channel has the highest/lowest churn rate?**

In [ ]:
df.groupby('acquisition_channel')['churned'].mean().sort_values(ascending=False)

An interesting finding is that, although Google Ads acquires the largest number of customers, it also has the highest churn rate. One possible explanation is that Google Ads reaches a broad audience, attracting users who may be exploring the service out of curiosity rather than having a strong intent to stay, which could result in higher churn.

Referral and Organic channels have the lowest churn rates. A possible reason is that customers acquired through referrals often have greater trust in the service becuse of recommendations from existing users.
Organic customers are more likely to have actively searched for the service because they already had a genuine need. This may lead to higher customer retention and lower churn rates.

**How do customer behavior and engagement differ between churned and retained customers?**

In [ ]:
df.groupby('churned')[['active_days', 'last_login_days_ago','feat_usage_score_percent', 'support_tickets']].mean()

1. Active Days: There is very little difference in the average number of active days between churned and retained customers, suggesting that this feature is not a strong indicator of churn.
2. Last Login Days Ago: Churned customers have not logged in for much longer on average (36.3 days vs. 16.7 days), indicating that customers who stop using the platform are more likely to churn.
3. Feature Usage Score: Retained customers have a much higher average feature usage score (0.62 vs. 0.32), suggesting that customers who actively use the product's features are more likely to stay.
4. Support Tickets: Churned customers submitted more than twice as many support tickets on average (6.2 vs. 2.9). This may indicate that customers who experience more issues or require more support are more likely to leave the service.



# Predicting Customer Churn

## Logistic Regression

I built a Logistic Regression model to predict whether a customer will churn or not. First, I selected the relevant features and the target variable (churned). Then, I separated the numerical and categorical features, standardized the numerical variables, and one-hot encoded the categorical variables to prepare the data for modeling.

Next, I split the dataset into training (80%) and testing (20%) sets and trained the Logistic Regression model. at last, I evaluated its performance using accuracy, precision, recall, F1-score, and a confusion matrix.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Defining features (X) and target (y)
X = df[['industry', 'country', 'plan_level', 'billing_cycle', 'feat_usage_score_percent', 'acquisition_channel', 'last_login_days_ago', 'nps_score', 'log_monthly_revenue', 'log_support_tickets', 'active_days_clipped']]
y = df['churned']

# Identifying numerical and categorical columns
numerical_cols = ['feat_usage_score_percent', 'last_login_days_ago', 'nps_score', 'log_monthly_revenue', 'log_support_tickets', 'active_days_clipped']
categorical_cols = ['industry', 'country', 'plan_level', 'billing_cycle', 'acquisition_channel']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Applying preprocessing to training and test data
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# liblinear for L1/L2 regularization
log_reg_model = LogisticRegression(random_state=42, solver='liblinear')
log_reg_model.fit(X_train_scaled, y_train)

# Predictions on the test set
y_pred_log_reg = log_reg_model.predict(X_test_scaled)

# Evaluate the Logistic Regression model
print("\nLogistic Regression Classifier Results:\n")
print(classification_report(y_test, y_pred_log_reg))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_log_reg))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_log_reg)
plt.title("Logistic Regression Confusion Matrix")
plt.show()

Results:

The model achieved an accuracy of 95%. However, in churn prediction, accuracy alone is not sufficient because a model can achieve high accuracy while still failing to correctly identify customers who are at risk of churning. Therefore, I also evaluated precision, recall, and F1-score. All metrics are around 94–96%, indicating that the model performs well for both retained and churned customers. The confusion matrix shows that the model correctly classified 138 retained customers and 90 churned customers, making 13 misclassifications (6 false positives (not churned but model says churned) and 7 false negatives (churned but model says otherwise)).

**Feature Importance**

In [ ]:
# Getting feature names after preprocessing
feature_names = [
    name.replace('num__', '').replace('cat__', '')
    for name in preprocessor.get_feature_names_out()
]

coefficients = log_reg_model.coef_[0]

# Creating feature importance df
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)})

# Sortingg by absolute importance
feature_importance = feature_importance.sort_values(
    by='Abs_Coefficient',
    ascending=False)

# Displaing top features
print("\nFeature Importances:\n")
print(feature_importance[['Feature', 'Coefficient']].head(20))

I examined the feature coefficients to understand which variables had the greatest influence on churn. Positive coefficients indicate that a feature increases the likelihood of churn, while negative coefficients indicate that it reduces the likelihood of churn.